# BioMed Research Assistant - Kaggle GPU Training

This notebook is specifically configured to run the generic multi-domain data ingestion, model processing, and summarization using Kaggle's free GPU resources.

### Instructions before running:
1. **Enable GPU**: Go to the right pane (`Settings` → `Accelerator` → Select `GPU P100` or `GPU T4 x2`).
2. **Enable Internet**: Ensure `Internet` is turned ON in the settings so the script can fetch papers from NCBI and download HuggingFace models.
3. **Dataset Access**: You need your source code here. You can either zip your `BioMed_ResearchHelper` folder (excluding `venv/` and `cache/`) and upload it via `+ Add Data` as a dataset, **or** you can clone it via Git if it's hosted online.

In [ ]:
import os

# IF YOU UPLOADED YOUR PROJECT AS A DATASET called 'biomed-src':
dataset_path = "/kaggle/input/biomed-src"

if os.path.exists(dataset_path):
    print("Found dataset. Copying to writeable directory...")
    !cp -r {dataset_path}/* /kaggle/working/
    os.chdir("/kaggle/working/")
else:
    print("Dataset not found. We will assume you are manually placing files in /kaggle/working/")
    os.chdir("/kaggle/working/")

### 1. Set Environment Variables
Since `.env` files are typically ignored in uploads / git, you can run this cell to securely write your configuration on Kaggle.

In [ ]:
env_content = """NCBI_EMAIL=darainhyder21@gmail.com
NCBI_API_KEY=c13ca15722c4d8da43acd903d68e06497e08

RESEARCH_DOMAINS=CRISPR gene editing therapy,mRNA vaccine development,cancer immunotherapy CAR-T,microbiome gut brain axis,precision medicine personalized oncology,artificial intelligence medical imaging,neurodegeneration alzheimers biomarkers,antimicrobial resistance superbugs,epigenetics aging longevity,nanomedicine targeted drug delivery,regenerative medicine stem cells,synthetic biology biosensors,immunometabolism autoimmune disease,wearable biosensors digital health
PAPERS_PER_DOMAIN=500
DATE_FROM=2018/01/01
DATE_TO=2026/12/31

EMBEDDING_MODEL=pritamdeka/S-PubMedBert-MS-MARCO
SUMMARIZATION_MODEL=facebook/bart-large-cnn

DATA_DIR=./data
MODELS_DIR=./models
CACHE_DIR=./cache
"""

with open('.env', 'w') as f:
    f.write(env_content)
print(".env generated successfully!")

### 2. Install Dependencies
Install the required python packages from `requirements.txt`.

In [ ]:
# Kaggle already has numpy, pandas, and PyTorch optimized for its GPUs.
# Strict version pinning in requirements.txt breaks Kaggle's environment, so we remove the '==' constraints.
with open('requirements.txt', 'r') as f:
    reqs = f.read().replace('==', '>=')
with open('requirements.txt', 'w') as f:
    f.write(reqs)

!pip install -r requirements.txt

### 3. Run Pipeline
Run the generic multi-domain data ingestion, model processing, vector index building, and topic modeling.

In [ ]:
# This will take a while (hours potentially) depending on PAPERS_PER_DOMAIN
!python main.py setup

### 4. Pack & Export Trained Data
Zip the resulting `data/` output (which holds your models, vectors, processed JSONL, and topics) so you can download it directly from Kaggle and plug it back into your local machine.

In [ ]:
import shutil
import os
from IPython.display import FileLink

print("Zipping data folder...")
shutil.make_archive('trained_biomed_data', 'zip', 'data')
print("Done! You can download the trained_biomed_data.zip from the '/kaggle/working/' folder in the right pane.")

FileLink(r'trained_biomed_data.zip')